In [104]:
# pip install kagglehub[pandas-datasets]
import re
import ast
import unicodedata
import os
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "himanshupoddar/zomato-bangalore-restaurants",
    "zomato.csv"
)

print(df.head())

/tmp/ipykernel_58/3645472393.py:11: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


                                                 url  \
0  https://www.zomato.com/bangalore/jalsa-banasha...   
1  https://www.zomato.com/bangalore/spice-elephan...   
2  https://www.zomato.com/SanchurroBangalore?cont...   
3  https://www.zomato.com/bangalore/addhuri-udupi...   
4  https://www.zomato.com/bangalore/grand-village...   

                                             address                   name  \
0  942, 21st Main Road, 2nd Stage, Banashankari, ...                  Jalsa   
1  2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...         Spice Elephant   
2  1112, Next to KIMS Medical College, 17th Cross...        San Churro Cafe   
3  1st Floor, Annakuteera, 3rd Stage, Banashankar...  Addhuri Udupi Bhojana   
4  10, 3rd Floor, Lakshmi Associates, Gandhi Baza...          Grand Village   

  online_order book_table   rate  votes                             phone  \
0          Yes        Yes  4.1/5    775    080 42297555\r\n+91 9743772233   
1          Yes         No  4.1/5  

In [105]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51717 entries, 0 to 51716
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   url                          51717 non-null  object
 1   address                      51717 non-null  object
 2   name                         51717 non-null  object
 3   online_order                 51717 non-null  object
 4   book_table                   51717 non-null  object
 5   rate                         43942 non-null  object
 6   votes                        51717 non-null  int64 
 7   phone                        50509 non-null  object
 8   location                     51696 non-null  object
 9   rest_type                    51490 non-null  object
 10  dish_liked                   23639 non-null  object
 11  cuisines                     51672 non-null  object
 12  approx_cost(for two people)  51371 non-null  object
 13  reviews_list                 51

In [106]:
df.describe(include='all')

,url,address,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,cuisines,approx_cost(for two people),reviews_list,menu_item,listed_in(type),listed_in(city)
count,51717,51717,51717,51717,51717,43942,51717.000000,50509,51696,51490,23639,51672,51371,51717,51717,51717,51717
unique,51717,11495,8792,2,2,64,NaN,14926,93,93,5271,2723,70,22513,9098,7,30
top,https://www.zomato.com/bangalore/the-nest-the-...,Delivery Only,Cafe Coffee Day,Yes,No,NEW,NaN,080 43334321,BTM,Quick Bites,Biryani,North Indian,300,[],[],Delivery,BTM
freq,1,128,96,30444,45268,2208,NaN,216,5124,19132,182,2913,7576,7595,39617,25942,3279
mean,NaN,NaN,NaN,NaN,NaN,NaN,283.697527,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,803.838853,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,7.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,41.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,198.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [107]:
df.columns

Index(['url', 'address', 'name', 'online_order', 'book_table', 'rate', 'votes',
       'phone', 'location', 'rest_type', 'dish_liked', 'cuisines',
       'approx_cost(for two people)', 'reviews_list', 'menu_item',
       'listed_in(type)', 'listed_in(city)'],
      dtype='object')

In [108]:
df.isnull().sum()

url                                0
address                            0
name                               0
online_order                       0
book_table                         0
rate                            7775
votes                              0
phone                           1208
location                          21
rest_type                        227
dish_liked                     28078
cuisines                          45
approx_cost(for two people)      346
reviews_list                       0
menu_item                          0
listed_in(type)                    0
listed_in(city)                    0
dtype: int64

In [109]:
df.duplicated().sum()

np.int64(0)

In [110]:
drop_cols = [
    "url",
    "address",
    "phone"
]

df.drop(columns=drop_cols, inplace=True)

In [111]:
df.shape

(51717, 14)

In [112]:
# Clean Name Column

import numpy as np

# Convert to string
df["name"] = df["name"].astype(str)

# Remove leading/trailing spaces
df["name"] = df["name"].str.strip()

# Replace newlines with spaces
df["name"] = df["name"].str.replace("\n", " ", regex=False)

# Remove multiple spaces
df["name"] = df["name"].str.replace(r"\s+", " ", regex=True)

# Remove unwanted quotes at the beginning/end
df["name"] = df["name"].str.strip("'\"")

# Fix common encoding issue
df["name"] = df["name"].str.replace("Ã", "", regex=False)

# Replace invalid names with NaN instead of dropping immediately
invalid_pattern = (
    r"^(Rated|RATED)|"
    r"^(service|taste|ambience|quantity|ordered|delivery|review)$"
)

df.loc[
    df["name"].str.contains(invalid_pattern, case=False, na=False),
    "name"
] = np.nan

# Replace very short names (less than 2 characters) with NaN
df.loc[df["name"].str.len() < 2, "name"] = np.nan

# Replace very long names (more than 80 characters) with NaN
df.loc[df["name"].str.len() > 80, "name"] = np.nan

# Drop only rows where name is missing
df.dropna(subset=["name"], inplace=True)

# Reset index
df.reset_index(drop=True, inplace=True)

print(df.shape)
df.head()

(51709, 14)


/tmp/ipykernel_58/3640817925.py:30: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["name"].str.contains(invalid_pattern, case=False, na=False),


,name,online_order,book_table,rate,votes,location,rest_type,dish_liked,cuisines,approx_cost(for two people),reviews_list,menu_item,listed_in(type),listed_in(city)
0,Jalsa,Yes,Yes,4.1/5,775,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800,"[('Rated 4.0', 'RATED\n A beautiful place to ...",[],Buffet,Banashankari
1,Spice Elephant,Yes,No,4.1/5,787,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800,"[('Rated 4.0', 'RATED\n Had been here for din...",[],Buffet,Banashankari
2,San Churro Cafe,Yes,No,3.8/5,918,Banashankari,"Cafe, Casual Dining","Churros, Cannelloni, Minestrone Soup, Hot Choc...","Cafe, Mexican, Italian",800,"[('Rated 3.0', ""RATED\n Ambience is not that ...",[],Buffet,Banashankari
3,Addhuri Udupi Bhojana,No,No,3.7/5,88,Banashankari,Quick Bites,Masala Dosa,"South Indian, North Indian",300,"[('Rated 4.0', ""RATED\n Great food and proper...",[],Buffet,Banashankari
4,Grand Village,No,No,3.8/5,166,Basavanagudi,Casual Dining,"Panipuri, Gol Gappe","North Indian, Rajasthani",600,"[('Rated 4.0', 'RATED\n Very good restaurant ...",[],Buffet,Banashankari


In [113]:
df.shape

(51709, 14)

In [114]:
# Clean Online Order Column 
print(df["online_order"].unique())
print(df["online_order"].value_counts(dropna=False))

['Yes' 'No']
online_order
Yes    30437
No     21272
Name: count, dtype: int64


In [115]:
# Convert to string
df["online_order"] = df["online_order"].astype(str)

# Remove leading/trailing spaces
df["online_order"] = df["online_order"].str.strip()

# Standardize case
df["online_order"] = df["online_order"].str.title()

# Replace common missing values with NaN
df["online_order"] = df["online_order"].replace(
    ["Nan", "None", "N/A", "-", ""],
    pd.NA
)

# Keep only valid values (Yes/No)
df = df[df["online_order"].isin(["Yes", "No"])]

# Reset index
df.reset_index(drop=True, inplace=True)

In [116]:
print(df["online_order"].value_counts())
print(df["online_order"].unique())


online_order
Yes    30437
No     21272
Name: count, dtype: int64
['Yes' 'No']


In [117]:
df.shape

(51709, 14)

In [118]:
# Clean book table Column 
print(df["book_table"].unique())
print(df["book_table"].value_counts(dropna=False))

['Yes' 'No']
book_table
No     45261
Yes     6448
Name: count, dtype: int64


In [119]:
# Convert to string
df["book_table"] = df["book_table"].astype(str)

# Remove leading/trailing spaces
df["book_table"] = df["book_table"].str.strip()

# Standardize case
df["book_table"] = df["book_table"].str.title()

# Replace common missing values with NaN
df["book_table"] = df["book_table"].replace(
    ["Nan", "None", "N/A", "-", ""],
    pd.NA
)

# Keep only valid values (Yes/No)
df = df[df["book_table"].isin(["Yes", "No"])]

# Reset index
df.reset_index(drop=True, inplace=True)

In [120]:
print(df["book_table"].unique())
print(df["book_table"].value_counts())


['Yes' 'No']
book_table
No     45261
Yes     6448
Name: count, dtype: int64


In [121]:
# Rate column cleaning 
import pandas as pd
import numpy as np

# -----------------------------
# Clean Rate Column
# -----------------------------

# Convert to string
df["rate"] = df["rate"].astype(str).str.strip()

# Replace common missing values with NaN
df["rate"] = df["rate"].replace(
    [
        "",
        " ",
        "NEW",
        "new",
        "nan",
        "NaN",
        "None",
        "NULL",
        "null",
        "<NA>"
    ],
    np.nan
)

# Extract only rating like 4.1 or 3.8 from any text
df["rate"] = df["rate"].str.extract(r'([0-5](?:\.\d)?)')

# Convert to numeric
df["rate"] = pd.to_numeric(df["rate"], errors="coerce")

# Keep only ratings between 0 and 5
df.loc[(df["rate"] < 0) | (df["rate"] > 5), "rate"] = np.nan

# -----------------------------
# Make 5-10% values equal to 0
# -----------------------------

target_zero_percent = 0.07      # 7% zeros (change to 0.05-0.10 if desired)

total_rows = len(df)
target_zero_count = int(total_rows * target_zero_percent)

# Current missing values
missing_idx = df[df["rate"].isna()].index

# If missing values are more than target
if len(missing_idx) >= target_zero_count:

    # Randomly choose rows to become 0
    zero_idx = np.random.choice(
        missing_idx,
        size=target_zero_count,
        replace=False
    )

    df.loc[zero_idx, "rate"] = 0

    # Fill remaining missing values with median rating
    median_rating = df["rate"].median()
    df["rate"] = df["rate"].fillna(median_rating)

else:
    # If missing values are fewer than target
    df.loc[missing_idx, "rate"] = 0

# Round to one decimal place
df["rate"] = df["rate"].round(1)

# Final datatype
df["rate"] = df["rate"].astype(float)

# -----------------------------
# Check Results
# -----------------------------
print(df["rate"].describe())

print("\nUnique Ratings:")
print(sorted(df["rate"].unique()))

print("\nPercentage of Zero Ratings:")
print((df["rate"] == 0).mean() * 100)

count    51709.000000
mean         3.441368
std          1.023522
min          0.000000
25%          3.400000
50%          3.700000
75%          3.900000
max          4.900000
Name: rate, dtype: float64

Unique Ratings:
[np.float64(0.0), np.float64(1.8), np.float64(2.0), np.float64(2.1), np.float64(2.2), np.float64(2.3), np.float64(2.4), np.float64(2.5), np.float64(2.6), np.float64(2.7), np.float64(2.8), np.float64(2.9), np.float64(3.0), np.float64(3.1), np.float64(3.2), np.float64(3.3), np.float64(3.4), np.float64(3.5), np.float64(3.6), np.float64(3.7), np.float64(3.8), np.float64(3.9), np.float64(4.0), np.float64(4.1), np.float64(4.2), np.float64(4.3), np.float64(4.4), np.float64(4.5), np.float64(4.6), np.float64(4.7), np.float64(4.8), np.float64(4.9)]

Percentage of Zero Ratings:
6.998781643427643


In [122]:
df.shape

(51709, 14)

In [123]:
df

,name,online_order,book_table,rate,votes,location,rest_type,dish_liked,cuisines,approx_cost(for two people),reviews_list,menu_item,listed_in(type),listed_in(city)
0,Jalsa,Yes,Yes,4.1,775,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800,"[('Rated 4.0', 'RATED\n A beautiful place to ...",[],Buffet,Banashankari
1,Spice Elephant,Yes,No,4.1,787,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800,"[('Rated 4.0', 'RATED\n Had been here for din...",[],Buffet,Banashankari
2,San Churro Cafe,Yes,No,3.8,918,Banashankari,"Cafe, Casual Dining","Churros, Cannelloni, Minestrone Soup, Hot Choc...","Cafe, Mexican, Italian",800,"[('Rated 3.0', ""RATED\n Ambience is not that ...",[],Buffet,Banashankari
3,Addhuri Udupi Bhojana,No,No,3.7,88,Banashankari,Quick Bites,Masala Dosa,"South Indian, North Indian",300,"[('Rated 4.0', ""RATED\n Great food and proper...",[],Buffet,Banashankari
4,Grand Village,No,No,3.8,166,Basavanagudi,Casual Dining,"Panipuri, Gol Gappe","North Indian, Rajasthani",600,"[('Rated 4.0', 'RATED\n Very good restaurant ...",[],Buffet,Banashankari
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51704,Best Brews - Four Points by Sheraton Bengaluru...,No,No,3.6,27,Whitefield,Bar,NaN,Continental,"1,500","[('Rated 5.0', ""RATED\n Food and service are ...",[],Pubs and bars,Whitefield
51705,Vinod Bar And Restaurant,No,No,3.7,0,Whitefield,Bar,NaN,Finger Food,600,[],[],Pubs and bars,Whitefield
51706,Plunge - Sheraton Grand Bengaluru Whitefield H...,No,No,0.0,0,Whitefield,Bar,NaN,Finger Food,"2,000",[],[],Pubs and bars,Whitefield
51707,Chime - Sheraton Grand Bengaluru Whitefield Ho...,No,Yes,4.3,236,"ITPL Main Road, Whitefield",Bar,"Cocktails, Pizza, Buttermilk",Finger Food,"2,500","[('Rated 4.0', 'RATED\n Nice and friendly pla...",[],Pubs and bars,Whitefield


In [124]:
# Votes column Cleaning

# Convert to string
df["votes"] = df["votes"].astype(str).str.strip()

# Keep only values that are purely digits
mask = df["votes"].str.fullmatch(r"\d+")

# Invalid values become NaN
df.loc[~mask, "votes"] = np.nan

# Convert to numeric
df["votes"] = pd.to_numeric(df["votes"], errors="coerce")

# Fill missing values with median
median_votes = int(df["votes"].median())
df["votes"] = df["votes"].fillna(median_votes)

# Integer datatype
df["votes"] = df["votes"].astype(int)

print(df["votes"].describe())

count    51709.000000
mean       283.705235
std        803.895150
min          0.000000
25%          7.000000
50%         41.000000
75%        198.000000
max      16832.000000
Name: votes, dtype: float64


In [125]:
df

,name,online_order,book_table,rate,votes,location,rest_type,dish_liked,cuisines,approx_cost(for two people),reviews_list,menu_item,listed_in(type),listed_in(city)
0,Jalsa,Yes,Yes,4.1,775,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800,"[('Rated 4.0', 'RATED\n A beautiful place to ...",[],Buffet,Banashankari
1,Spice Elephant,Yes,No,4.1,787,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800,"[('Rated 4.0', 'RATED\n Had been here for din...",[],Buffet,Banashankari
2,San Churro Cafe,Yes,No,3.8,918,Banashankari,"Cafe, Casual Dining","Churros, Cannelloni, Minestrone Soup, Hot Choc...","Cafe, Mexican, Italian",800,"[('Rated 3.0', ""RATED\n Ambience is not that ...",[],Buffet,Banashankari
3,Addhuri Udupi Bhojana,No,No,3.7,88,Banashankari,Quick Bites,Masala Dosa,"South Indian, North Indian",300,"[('Rated 4.0', ""RATED\n Great food and proper...",[],Buffet,Banashankari
4,Grand Village,No,No,3.8,166,Basavanagudi,Casual Dining,"Panipuri, Gol Gappe","North Indian, Rajasthani",600,"[('Rated 4.0', 'RATED\n Very good restaurant ...",[],Buffet,Banashankari
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51704,Best Brews - Four Points by Sheraton Bengaluru...,No,No,3.6,27,Whitefield,Bar,NaN,Continental,"1,500","[('Rated 5.0', ""RATED\n Food and service are ...",[],Pubs and bars,Whitefield
51705,Vinod Bar And Restaurant,No,No,3.7,0,Whitefield,Bar,NaN,Finger Food,600,[],[],Pubs and bars,Whitefield
51706,Plunge - Sheraton Grand Bengaluru Whitefield H...,No,No,0.0,0,Whitefield,Bar,NaN,Finger Food,"2,000",[],[],Pubs and bars,Whitefield
51707,Chime - Sheraton Grand Bengaluru Whitefield Ho...,No,Yes,4.3,236,"ITPL Main Road, Whitefield",Bar,"Cocktails, Pizza, Buttermilk",Finger Food,"2,500","[('Rated 4.0', 'RATED\n Nice and friendly pla...",[],Pubs and bars,Whitefield


In [126]:
df.isnull().sum()

name                               0
online_order                       0
book_table                         0
rate                               0
votes                              0
location                          21
rest_type                        227
dish_liked                     28075
cuisines                          45
approx_cost(for two people)      346
reviews_list                       0
menu_item                          0
listed_in(type)                    0
listed_in(city)                    0
dtype: int64

In [127]:

# -----------------------------
# Clean Location Column
# -----------------------------

# Convert to string
df["location"] = df["location"].astype(str).str.strip()

# Replace common missing values
df["location"] = df["location"].replace(
    [
        "",
        " ",
        "nan",
        "NaN",
        "None",
        "NULL",
        "null",
        "<NA>"
    ],
    np.nan
)

# Standardize capitalization
df["location"] = df["location"].str.title()

# -----------------------------
# Valid Bangalore Locations
# -----------------------------

valid_locations = [
    'Banashankari','Basavanagudi','Mysore Road','Jayanagar',
    'Kumaraswamy Layout','Rajarajeshwari Nagar','Vijay Nagar',
    'Uttarahalli','Jp Nagar','South Bangalore','City Market',
    'Nagarbhavi','Bannerghatta Road','Btm','Kanakapura Road',
    'Bommanahalli'
]

# Keep only valid locations
df.loc[~df["location"].isin(valid_locations), "location"] = np.nan

# Fill missing values with most frequent location
mode_location = df["location"].mode()[0]

df["location"] = df["location"].fillna(mode_location)

# Convert to category (recommended)
df["location"] = df["location"].astype("category")

# -----------------------------
# Check Results
# -----------------------------
print(df["location"].value_counts())

print("\nMissing Values:", df["location"].isna().sum())

print("\nUnique Locations:", df["location"].nunique())

location
Btm                     43513
Jp Nagar                 2235
Jayanagar                1926
Bannerghatta Road        1630
Banashankari              906
Basavanagudi              684
Bommanahalli              238
Kumaraswamy Layout        195
City Market               126
South Bangalore           107
Vijay Nagar                80
Mysore Road                22
Kanakapura Road            19
Uttarahalli                17
Nagarbhavi                  9
Rajarajeshwari Nagar        2
Name: count, dtype: int64

Missing Values: 0

Unique Locations: 16


In [128]:
df.isnull().sum()

name                               0
online_order                       0
book_table                         0
rate                               0
votes                              0
location                           0
rest_type                        227
dish_liked                     28075
cuisines                          45
approx_cost(for two people)      346
reviews_list                       0
menu_item                          0
listed_in(type)                    0
listed_in(city)                    0
dtype: int64

In [129]:
df.shape

(51709, 14)

In [130]:
import pandas as pd

valid_rest_types = [
    "Quick Bites","Casual Dining","Cafe","Delivery","Dessert Parlor",
    "Bakery","Takeaway","Takeaway, Delivery","Mess","Sweet Shop",
    "Beverage Shop","Pub","Bar","Lounge","Fine Dining","Food Court",
    "Food Truck","Kiosk","Confectionery","Microbrewery","Dhaba","Club",
    "Irani Cafee"
]

df["rest_type"] = (
    df["rest_type"]
    .astype(str)
    .str.strip()
    .replace(["", "nan", "Nan", "None", "N/A", "-", "NULL"], pd.NA)
)

pattern = r"Rated|RATED|\(|\)|\"|'|\n"
df.loc[df["rest_type"].str.contains(pattern, case=False, na=False), "rest_type"] = pd.NA

def clean_rest_type(x):
    if pd.isna(x):
        return pd.NA
    parts = [p.strip() for p in str(x).split(",")]
    parts = [p for p in parts if p in valid_rest_types]
    return ", ".join(parts) if parts else pd.NA

df["rest_type"] = df["rest_type"].apply(clean_rest_type)
df["rest_type"] = df["rest_type"].str.replace(r"\s*,\s*", ", ", regex=True).str.strip()

print(df["rest_type"].value_counts(dropna=False))

rest_type
Quick Bites                   19136
Casual Dining                 10330
Cafe                           3729
Delivery                       2603
Dessert Parlor                 2262
                              ...  
Bakery, Food Court                2
Dessert Parlor, Food Court        2
Food Court, Beverage Shop         2
Sweet Shop, Dessert Parlor        1
Quick Bites, Kiosk                1
Name: count, Length: 91, dtype: int64


In [131]:
df.shape

(51709, 14)

In [132]:
import pandas as pd

df["dish_liked"] = df["dish_liked"].astype(str).str.strip()

# Missing values
df["dish_liked"] = df["dish_liked"].replace(
    ["", "nan", "None", "NaN", "N/A", "-"],
    pd.NA
)

# Remove ratings
df["dish_liked"] = df["dish_liked"].str.replace(
    r".*Rated\s*\d+(\.\d+)?['\"]?\)?",
    "",
    regex=True
)

# Remove review text
df["dish_liked"] = df["dish_liked"].str.replace(
    r"(?is)^.*RATED.*",
    "",
    regex=True
)

# Remove encoding issues
df["dish_liked"] = df["dish_liked"].str.replace(
    r"[ÃÂ�]+",
    "",
    regex=True
)

# Remove newlines
df["dish_liked"] = df["dish_liked"].str.replace(r"\n", " ", regex=True)

# Remove long reviews
def clean_dishes(x):
    if pd.isna(x):
        return pd.NA

    x = str(x).strip()

    if len(x.split()) > 25:
        return pd.NA

    if any(ch in x for ch in [".", "!", "?"]):
        return pd.NA

    return x

df["dish_liked"] = df["dish_liked"].apply(clean_dishes)

# Remove duplicate commas
df["dish_liked"] = (
    df["dish_liked"]
    .str.replace(r",\s*,", ", ", regex=True)
    .str.strip(", ")
)

# Convert empty strings to NaN
df["dish_liked"] = df["dish_liked"].replace("", pd.NA)

In [133]:
df.shape

(51709, 14)

In [134]:
import pandas as pd
import numpy as np

# Convert to string
df["cuisines"] = df["cuisines"].astype(str).str.strip()

# Replace fake missing values
df["cuisines"] = df["cuisines"].replace(
    ["", "nan", "None", "NaN", "N/A"],
    np.nan
)

# Remove review text
mask = (
    df["cuisines"].str.contains("rated", case=False, na=False) |
    df["cuisines"].str.contains("\n", na=False)
)
df.loc[mask, "cuisines"] = np.nan

# Remove quotes and brackets
df["cuisines"] = (
    df["cuisines"]
      .str.replace("'", "", regex=False)
      .str.replace('"', "", regex=False)
      .str.replace("(", "", regex=False)
      .str.replace(")", "", regex=False)
)

# Remove duplicate cuisines
def remove_duplicate_cuisines(text):
    if pd.isna(text):
        return text

    cuisines = [x.strip() for x in text.split(",")]
    cuisines = list(dict.fromkeys(cuisines))
    return ", ".join(cuisines)

df["cuisines"] = df["cuisines"].apply(remove_duplicate_cuisines)

# Standardize spacing
df["cuisines"] = (
    df["cuisines"]
      .str.replace(r"\s*,\s*", ", ", regex=True)
      .str.strip(", ")
)

# Standardize capitalization
df["cuisines"] = df["cuisines"].apply(
    lambda x: ", ".join(i.strip().title() for i in x.split(","))
    if pd.notna(x) else x
)

# Keep only valid cuisine strings
pattern = r"^[A-Za-z&\-\s,]+$"
df.loc[~df["cuisines"].str.match(pattern, na=False), "cuisines"] = np.nan

In [135]:
df.shape

(51709, 14)

In [136]:
import pandas as pd

# Rename column
df.rename(columns={
    "approx_cost(for two people)": "approx_cost"
}, inplace=True)

# Convert to string
df["approx_cost"] = df["approx_cost"].astype(str)

# Remove commas
df["approx_cost"] = df["approx_cost"].str.replace(",", "", regex=False)

# Remove spaces
df["approx_cost"] = df["approx_cost"].str.strip()

# Convert valid numbers, invalid values become NaN
df["approx_cost"] = pd.to_numeric(df["approx_cost"], errors="coerce")

# Fill missing values with median
df["approx_cost"] = df["approx_cost"].fillna(df["approx_cost"].median())

# Convert to integer
df["approx_cost"] = df["approx_cost"].astype(int)

# Check result
print(df["approx_cost"].head())
print(df["approx_cost"].describe())
print(df["approx_cost"].isna().sum())

0    800
1    800
2    800
3    300
4    600
Name: approx_cost, dtype: int64
count    51709.000000
mean       554.373997
std        437.563776
min         40.000000
25%        300.000000
50%        400.000000
75%        650.000000
max       6000.000000
Name: approx_cost, dtype: float64
0


In [137]:
df.shape

(51709, 14)

In [138]:
import ast
import pandas as pd

def clean_menu_item(x):
    if pd.isna(x):
        return []

    # Convert string representation of list
    if isinstance(x, str):
        x = x.strip()
        if x == "[]":
            return []
        try:
            x = ast.literal_eval(x)
        except:
            return []

    if not isinstance(x, list):
        return []

    cleaned = []
    for item in x:
        item = str(item).replace("\n", " ").replace("\r", " ").strip()
        item = item.replace("Ã", "").replace("Â", "").replace("ƒ", "")
        item = " ".join(item.split())

        # Skip obvious non-menu content
        if not item:
            continue
        if "rated" in item.lower():
            continue
        if len(item) > 80:
            continue

        cleaned.append(item.title())

    # Remove duplicates while preserving order
    return list(dict.fromkeys(cleaned))

df["menu_item"] = df["menu_item"].apply(clean_menu_item)

In [139]:
df.isnull().sum()

name                   0
online_order           0
book_table             0
rate                   0
votes                  0
location               0
rest_type            232
dish_liked         28075
cuisines              45
approx_cost            0
reviews_list           0
menu_item              0
listed_in(type)        0
listed_in(city)        0
dtype: int64